# Parsing a capture into one table per event`tasks/parse_logs.yml` **is** the job. This notebook runs it, and then looks atwhat it did.The shape of it: one pass over a folder of trading logs, every line parsed intoan `Event`, classified by regular expression, and appended to the Iceberg tablefor what it is about — `order_logs`, `execution_logs`, … `unknown_logs`. Onepass, not one per kind; streaming, not staged; appending, so re-running it overa capture that grew costs the growth.

## A capture to readThe job in the YAML points at `../data/capture`, which is not in thisrepository — a capture is somebody's data, not ours. So this cell writes asmall one into a scratch folder and points the job at that instead. Everythingbelow is exactly what a real run does.

In [ ]:
import pathlibimport tempfilework = pathlib.Path(tempfile.mkdtemp(prefix="parse-logs-"))capture = work / "capture"capture.mkdir()#: One line per kind, so the capture holds a bit of everything a log does.KINDS = [    "8=FIX.4.4\x0135=8\x0117=e{i}\x01",                 # an execution report    "sent NewOrderSingle AAPL {i}@10.0",                  # an order    "8=FIX.4.4\x0135=X\x01268={i}\x01",                 # an incremental book update    "8=FIX.4.4\x0135=W\x01268={i}\x01",                 # a book snapshot    "8=FIX.4.4\x0135=S\x01117=q{i}\x01",                # a quote    "heartbeat {i}",                                      # and something nothing matches]def write_capture(name: str, day: str, rows: int = 600) -> int:    lines = [        f"{day} {(i // 300) % 24:02d}:{(i // 5) % 60:02d}:{i % 60:02d}.167_520 "        f"[t-1] [Bridge] " + KINDS[i % len(KINDS)].format(i=i)        for i in range(rows)    ]    (capture / name).write_text("\n".join(lines) + "\n")    return len(lines)written = write_capture("a.log", "2026-08-14")print(f"{written} lines in {capture}")print((capture / "a.log").read_text().splitlines()[0])

## The job, read from the file`Task.from_yaml` reads the document and dispatches on its `kind` — the same`from_yaml` that reads a schema contract. Nothing here knows what a `ParseLogs`is until the file says so.The two overrides are the scratch capture and a scratch catalog; every otherfield is what the file declared.

In [ ]:
from rekep import Tasktask = Task.from_yaml("parse_logs.yml")warehouse = work / "warehouse"warehouse.mkdir()task.source = str(capture)task.timezone = None                      # the sample is written in UTCtask.properties = {    "type": "sql",    "uri": f"sqlite:///{(work / 'catalog.db').as_posix()}",    "warehouse": warehouse.as_uri(),}print(type(task).__name__, "->", task.name)print("rules:  ", [rule.etype.name for rule in task.rules.rules])print("targets:", [task.target_name(int(rule.etype)) for rule in task.rules.rules])

## Running itOne pass. The report says what was read, what landed where, and what wasalready stored — returned rather than printed, so a scheduler can check it.

In [ ]:
report = task.run()print(report)report.written

## What landedEach table holds one kind, and nothing else. The line nothing matched is in`unknown_logs` — still parsed, still keyed, still partitioned. Dropping itwould make the job lossy exactly when a log format changes.

In [ ]:
from rekep.market import EventTypefor name, landed in sorted(report.written.items()):    code_ = next(k for k in EventType if task.target_name(int(k)) == name)    rows = task.target(int(code_)).read_arrow_table()    kinds = set(rows.column("etype").to_pylist())    print(f"{name:26} {rows.num_rows:5,} rows   etype={ {EventType(c).name for c in kinds} }")

In [ ]:
orders = task.target(int(EventType.ORDER)).read_arrow_table()shown = orders.select(["unix", "hunix", "etype", "hash", "message", "bridge"])#: `to_pandas()` here if pandas is about; a slice prints without it.shown.slice(0, 5).to_pylist()

## The envelopeA parsed line is an `Event` like everything else this package stores, which iswhat lets it be read beside the orders and books it describes:- `unix` is the instant the line is stamped with, and `hunix` is that hour —  what the table is partitioned on.- `hash` is the digest of the raw line, and it is half the primary key. The same  capture read twice deduplicates itself.- `xhash` is the same digest: a log line is one version of one thing and never  changes, so its lifecycle is itself.

In [ ]:
print("partitions:", sorted({v for v in orders.column("hunix").to_pylist()})[:4], "...")print("hash is 16 bytes:", orders.column("hash").type)print("xhash == hash:  ", orders.column("xhash").equals(orders.column("hash")))print()print(task.target(int(EventType.ORDER)).into_struct_field().partition_keys())

## Running it againThe point of appending with `merge_by`: a replay reads the capture and writesnothing. Nothing stored is rewritten, and no delete file is produced.

In [ ]:
print(task.run())

## A capture that grewWhich is the case a scheduled job is in every time but the first. Only the newday is paid for.

In [ ]:
grew = write_capture("b.log", "2026-08-15")after = task.run()print(after)print()print(f"read {after.rows:,}, landed {after.landed:,}, already stored {after.skipped:,}")assert after.landed == grew

## Cleaning up

In [ ]:
import shutilshutil.rmtree(work, ignore_errors=True)print("gone:", work)